In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
"""
Linear atau MLP Probe pada Cached SigLIP Embedding, Versi Dioptimalkan (v2)
Big Data Challenge Satria Data 2026

Versi ini memperbaiki masalah performa dari versi sebelumnya, TANPA
mengurangi N_AUG dan TANPA memindah data keluar dari Google Drive.

Perubahan utama dari versi sebelumnya (v1 -> v2):

1. NUM_WORKERS_EKSTRAKSI dinaikkan. Latency baca file dari Google Drive
   (via FUSE mount) itu per-request, bukan soal bandwidth. Menambah
   jumlah worker paralel membuat banyak request baca menumpuk
   bersamaan, sehingga latency per-file "disembunyikan" oleh
   concurrency, tanpa perlu memindahkan data ke disk lokal Colab.

2. Pemanggilan image processor SigLIP (resize, normalize, ke tensor)
   yang sebelumnya dipanggil di main process sekarang dipindah ke
   dalam Dataset.__getitem__, sehingga ikut berjalan paralel di
   worker CPU bersamaan dengan baca file. Sebelumnya bagian ini
   menumpuk di main process dan membuat GPU menunggu.

3. DataLoader diberi prefetch_factor lebih besar, supaya buffer batch
   yang sudah selesai diproses worker lebih dalam, mengurangi jeda
   GPU menunggu batch berikutnya.

4. torch.backends.cudnn.benchmark diaktifkan karena ukuran input
   konstan (384x384), cuDNN bisa memilih kernel tercepat secara
   otomatis.

Fitur dari versi sebelumnya tetap dipertahankan penuh:
- Aman dari OOM VRAM (batch dipecah otomatis dan dicoba ulang).
- Aman dari OOM RAM untuk gambar beresolusi sangat besar (downsize
  sebelum diproses).
- Checkpoint dan resume kalau runtime Colab disconnect di tengah
  proses ekstraksi.
"""

import os
import gc
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, Dataset
from PIL import Image
from torchvision import transforms
from tqdm import tqdm
from sklearn.metrics import f1_score, classification_report

# ============================================================
# KONFIGURASI, SEMUA DIATUR DI SINI
# ============================================================

ROOT_DIR = "/content/drive/MyDrive/BDC"

TRAIN_MANIFEST_PATH = f"{ROOT_DIR}/Preprocessing_Data/train_manifest.csv"
VAL_MANIFEST_PATH = f"{ROOT_DIR}/Preprocessing_Data/val_manifest.csv"
TEST_DIR_DRIVE = f"{ROOT_DIR}/test"

KOLOM_PATH_MANIFEST = "file_path"
KOLOM_LABEL_MANIFEST = "label"

CACHE_DIR = f"{ROOT_DIR}/EDA/cache_embedding_siglip"
MODEL_SAVE_DIR = f"{ROOT_DIR}/Train_model/Best_model"
SUBMISSION_PATH = f"{ROOT_DIR}/submission_linear_probe_siglip.csv"

SIGLIP_MODEL_NAME = "google/siglip-so400m-patch14-384"

N_AUG = 2
N_TTA = 5

BATCH_SIZE_EKSTRAKSI = 64

# Dinaikkan dari 2. Latency baca Google Drive itu per-request, jadi
# menambah worker paralel langsung mempercepat throughput baca tanpa
# perlu memindah data ke disk lokal. Kalau runtime Colab yang dipakai
# ternyata cuma kasih jatah CPU sedikit, turunkan lagi angka ini
# (cek dengan os.cpu_count(), lalu pakai sekitar itu, jangan jauh
# melebihi jumlah core yang tersedia).
NUM_WORKERS_EKSTRAKSI = 8

# Buffer batch yang sudah diproses worker, menunggu di-consume oleh
# GPU. Dinaikkan supaya GPU jarang menunggu batch berikutnya.
PREFETCH_FACTOR_EKSTRAKSI = 2

BATCH_SIZE_TRAINING = 128
EPOCHS = 30
LR = 1e-3
WEIGHT_DECAY = 1e-4
HIDDEN_DIM = 256
DROPOUT = 0.3
LABEL_SMOOTHING = 0.05
PATIENCE_EARLY_STOPPING = 8

USE_FP16 = True

# Batas sisi terpanjang gambar sebelum di-downsize, mencegah RAM habis
# untuk gambar beresolusi sangat besar (mis. beberapa file Electronic
# yang bisa sampai 18MB). SigLIP toh akan resize ke 384px secara internal,
# jadi menahan resolusi penuh di RAM sebelum itu cuma buang-buang memori.
MAX_SISI_GAMBAR = 1024

# Checkpoint disimpan tiap kelipatan jumlah gambar ini, supaya proses
# ekstraksi bisa dilanjutkan (bukan diulang dari nol) kalau runtime
# Colab disconnect di tengah jalan.
CHECKPOINT_SETIAP_N_GAMBAR = 2000

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

CACHE_TRAIN = f"{CACHE_DIR}/embedding_train_naug{N_AUG}.npz"
CACHE_VAL = f"{CACHE_DIR}/embedding_val.npz"
CACHE_TEST_CLEAN = f"{CACHE_DIR}/embedding_test_clean.npz"
CACHE_TEST_TTA = f"{CACHE_DIR}/embedding_test_tta{N_TTA}.npz"
BEST_MODEL_PATH = f"{MODEL_SAVE_DIR}/best_model_mlp_probe_siglip.pth"

LABEL_MAP = {"Recyclable": 0, "Electronic": 1, "Organic": 2}
IDX_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}
DAFTAR_LABEL_URUT = sorted(LABEL_MAP, key=lambda x: LABEL_MAP[x])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Ukuran input ke SigLIP selalu konstan (384x384), jadi cuDNN bisa
# mengunci kernel tercepat untuk ukuran itu alih-alih mencari ulang
# tiap kali bentuk berubah. Ini gratis, tidak ada trade-off di sini.
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True

print(f"Device {DEVICE}")
print(f"ROOT_DIR {ROOT_DIR}")
print(f"NUM_WORKERS_EKSTRAKSI dipakai, {NUM_WORKERS_EKSTRAKSI}")


# ============================================================
# Remap Path Manifest dari Windows Lokal ke Path Colab atau Drive
# ============================================================

PATH_LAMA_WINDOWS = r"d:\Bagas\Satria_Data_BDC\BDC 2026"
PATH_BARU_COLAB = ROOT_DIR
PERLU_REMAP_PATH = True


def remap_path_manifest_jika_perlu(manifest_path, kolom_path):
    if not PERLU_REMAP_PATH:
        return

    df_manifest = pd.read_csv(manifest_path)
    contoh_path = str(df_manifest[kolom_path].iloc[0])

    if PATH_LAMA_WINDOWS.lower() not in contoh_path.lower():
        print(f"Path pada {manifest_path} sudah tidak mengandung path Windows lama, remap dilewati")
        return

    print(f"Path Windows lokal terdeteksi pada {manifest_path}, melakukan remap ke {PATH_BARU_COLAB}")

    df_manifest[kolom_path] = (
        df_manifest[kolom_path]
        .str.replace(PATH_LAMA_WINDOWS, PATH_BARU_COLAB, regex=False, case=False)
        .str.replace("\\", "/", regex=False)
    )
    df_manifest.to_csv(manifest_path, index=False)
    print(f"Remap path selesai, file {manifest_path} sudah diperbarui")


remap_path_manifest_jika_perlu(TRAIN_MANIFEST_PATH, KOLOM_PATH_MANIFEST)
remap_path_manifest_jika_perlu(VAL_MANIFEST_PATH, KOLOM_PATH_MANIFEST)


# ============================================================
# Utilitas: Baca Gambar Aman Terhadap Resolusi Sangat Besar
# ============================================================

def baca_gambar_aman(path_gambar, ukuran_fallback=384):
    """
    Membaca gambar dan men-downsize dulu kalau sisi terpanjangnya melebihi
    MAX_SISI_GAMBAR, sebelum convert ke RGB. Mencegah RAM membengkak untuk
    gambar beresolusi sangat besar, terutama saat dibaca paralel oleh
    banyak worker DataLoader sekaligus.
    """
    try:
        with Image.open(path_gambar) as img:
            lebar, tinggi = img.size
            if max(lebar, tinggi) > MAX_SISI_GAMBAR:
                rasio = MAX_SISI_GAMBAR / max(lebar, tinggi)
                ukuran_baru = (int(lebar * rasio), int(tinggi * rasio))
                img = img.resize(ukuran_baru, Image.BILINEAR)
            return img.convert("RGB")
    except Exception:
        return Image.new("RGB", (ukuran_fallback, ukuran_fallback), color=(0, 0, 0))


# ============================================================
# Utilitas: Checkpoint dan Resume untuk Proses Ekstraksi
# ============================================================

def muat_checkpoint_jika_ada(cache_path, punya_label):
    """
    Mengecek apakah ada checkpoint sementara dari proses ekstraksi
    sebelumnya yang belum selesai. Kalau ada, kembalikan progress yang
    sudah tersimpan supaya proses bisa dilanjutkan, bukan diulang dari
    awal.
    """
    checkpoint_path = cache_path.replace(".npz", "_checkpoint.npz")
    if not os.path.exists(checkpoint_path):
        return checkpoint_path, [], ([] if punya_label else None), 0

    print(f"Checkpoint sementara ditemukan di {checkpoint_path}, melanjutkan dari sana...")
    data = np.load(checkpoint_path, allow_pickle=True)
    daftar_embedding = [data["embedding"]]
    daftar_label = list(data["label"]) if punya_label else None
    jumlah_selesai = int(data["jumlah_selesai"])
    print(f"Melanjutkan dari gambar ke-{jumlah_selesai}")
    return checkpoint_path, daftar_embedding, daftar_label, jumlah_selesai


def simpan_checkpoint(checkpoint_path, daftar_embedding, daftar_label, jumlah_selesai):
    embedding_gabung = np.concatenate(daftar_embedding, axis=0)
    if daftar_label is not None:
        np.savez(
            checkpoint_path, embedding=embedding_gabung,
            label=np.array(daftar_label), jumlah_selesai=jumlah_selesai,
        )
    else:
        np.savez(checkpoint_path, embedding=embedding_gabung, jumlah_selesai=jumlah_selesai)


def hapus_checkpoint_jika_ada(checkpoint_path):
    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)


# ============================================================
# Augmentasi untuk Ekstraksi Embedding Train dan TTA Test
# ============================================================

def dapatkan_augmentasi_train():
    return transforms.Compose([
        transforms.RandomResizedCrop(384, scale=(0.75, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
        transforms.RandomRotation(degrees=10),
    ])


def dapatkan_augmentasi_tta(indeks_view):
    if indeks_view == 0:
        return transforms.Compose([transforms.Resize((384, 384))])

    return transforms.Compose([
        transforms.Resize((420, 420)),
        transforms.RandomCrop(384),
        transforms.RandomHorizontalFlip(p=0.5 if indeks_view % 2 == 0 else 0.0),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
    ])


# ============================================================
# Dataset untuk Ekstraksi
#
# Dataset ini bertugas membaca file gambar (langsung dari Google Drive),
# menerapkan augmentasi kalau perlu, DAN menjalankan image processor
# SigLIP (resize ke 384, normalize, ke tensor) -- semuanya dijalankan
# di proses worker terpisah lewat DataLoader. Sebelumnya image
# processor dipanggil di main process sehingga jadi titik penyempitan;
# sekarang ikut paralel bersama baca file, jadi betul-betul tumpang
# tindih dengan komputasi GPU batch sebelumnya.
# ============================================================

class DatasetEkstraksi(Dataset):
    def __init__(self, daftar_path, daftar_label=None, transform=None, image_processor=None):
        self.daftar_path = daftar_path
        self.daftar_label = daftar_label
        self.transform = transform
        self.image_processor = image_processor

    def __len__(self):
        return len(self.daftar_path)

    def __getitem__(self, idx):
        img = baca_gambar_aman(self.daftar_path[idx])

        if self.transform:
            img = self.transform(img)

        pixel_values = self.image_processor(images=img, return_tensors="pt")["pixel_values"][0]

        label = self.daftar_label[idx] if self.daftar_label is not None else -1
        return pixel_values, label


def collate_tensor(batch):
    """
    Collate function untuk batch berisi tensor pixel_values yang sudah
    diproses image processor di worker, tinggal ditumpuk jadi satu
    tensor batch memakai torch.stack.
    """
    pixel_values = torch.stack([item[0] for item in batch], dim=0)
    daftar_label = [item[1] for item in batch]
    return pixel_values, daftar_label


def buat_dataloader_ekstraksi(dataset, shuffle=False):
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE_EKSTRAKSI,
        shuffle=shuffle,
        num_workers=NUM_WORKERS_EKSTRAKSI,
        collate_fn=collate_tensor,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(NUM_WORKERS_EKSTRAKSI > 0),
        prefetch_factor=(PREFETCH_FACTOR_EKSTRAKSI if NUM_WORKERS_EKSTRAKSI > 0 else None),
    )


# ============================================================
# Ekstraksi Tensor Aman Terhadap Perbedaan Versi Transformers
# ============================================================

def ekstrak_tensor_dari_output(out):
    if torch.is_tensor(out):
        return out
    if hasattr(out, "pooler_output") and out.pooler_output is not None:
        return out.pooler_output
    if hasattr(out, "last_hidden_state"):
        return out.last_hidden_state[:, 0, :]
    raise TypeError(f"Tidak tahu cara mengambil tensor dari tipe, {type(out)}")


# ============================================================
# Memuat SigLIP, Dibekukan Penuh
# ============================================================

def muat_siglip():
    from transformers import SiglipModel, SiglipProcessor
    print(f"Memuat {SIGLIP_MODEL_NAME}")

    processor = SiglipProcessor.from_pretrained(SIGLIP_MODEL_NAME)
    dtype = torch.float16 if (USE_FP16 and DEVICE.type == "cuda") else torch.float32
    model = SiglipModel.from_pretrained(SIGLIP_MODEL_NAME, torch_dtype=dtype)
    model.eval()
    for param in model.parameters():
        param.requires_grad = False
    model.to(DEVICE)

    print("Vision encoder dibekukan")
    return model, processor


def hitung_embedding_batch_aman(model, batch_pixel_values, batch_size_minimum=1):
    """
    Menghitung embedding satu batch gambar. batch_pixel_values sudah
    berupa tensor hasil image processor (dikerjakan di worker), jadi
    di sini tinggal dipindah ke GPU dan dijalankan lewat model.

    Kalau ternyata VRAM tidak cukup untuk batch ini (CUDA out of
    memory), batch dipecah jadi dua bagian lebih kecil dan dicoba
    ulang secara rekursif, sampai berhasil atau sampai batas
    batch_size_minimum tercapai. Ini mencegah script berhenti total
    cuma karena satu batch kebetulan terlalu besar.
    """
    try:
        with torch.no_grad():
            pixel_values = batch_pixel_values.to(DEVICE)
            if USE_FP16 and DEVICE.type == "cuda":
                pixel_values = pixel_values.half()
            fitur = ekstrak_tensor_dari_output(model.get_image_features(pixel_values=pixel_values))
            fitur = fitur / fitur.norm(dim=-1, keepdim=True)
        return fitur.float().cpu().numpy()

    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        gc.collect()

        if len(batch_pixel_values) <= batch_size_minimum:
            raise RuntimeError(
                f"OOM bahkan pada batch berukuran {len(batch_pixel_values)}, "
                f"turunkan lagi BATCH_SIZE_EKSTRAKSI di konfigurasi"
            )

        print(f"OOM terdeteksi pada batch berukuran {len(batch_pixel_values)}, membagi dua dan mencoba ulang")
        titik_tengah = len(batch_pixel_values) // 2
        hasil_pertama = hitung_embedding_batch_aman(model, batch_pixel_values[:titik_tengah], batch_size_minimum)
        hasil_kedua = hitung_embedding_batch_aman(model, batch_pixel_values[titik_tengah:], batch_size_minimum)
        return np.concatenate([hasil_pertama, hasil_kedua], axis=0)


# ============================================================
# Ekstraksi Embedding Train dengan Augmentasi Berulang (n_aug)
# ============================================================

def ekstrak_embedding_train(df_manifest, model, image_processor, cache_path, n_aug):
    if os.path.exists(cache_path):
        print(f"Cache embedding train ditemukan, memuat dari {cache_path}")
        data = np.load(cache_path, allow_pickle=True)
        return data["embedding"], data["label"]

    print(f"Mengekstrak embedding train dengan n_aug {n_aug}")

    daftar_path_diulang = []
    daftar_label_diulang = []
    daftar_versi = []

    for _, baris in df_manifest.iterrows():
        for versi in range(n_aug):
            daftar_path_diulang.append(baris[KOLOM_PATH_MANIFEST])
            daftar_label_diulang.append(LABEL_MAP[baris[KOLOM_LABEL_MANIFEST]])
            daftar_versi.append(versi)

    total_semua = len(daftar_path_diulang)
    augmentasi_train = dapatkan_augmentasi_train()

    # --- Cek checkpoint, lanjutkan kalau ada sisa dari proses sebelumnya ---
    checkpoint_path, daftar_embedding, daftar_label_hasil, jumlah_selesai = muat_checkpoint_jika_ada(
        cache_path, punya_label=True
    )

    if jumlah_selesai >= total_semua:
        print("Seluruh data train (dengan augmentasi) sudah diproses sebelumnya dari checkpoint.")
    else:
        daftar_path_sisa = daftar_path_diulang[jumlah_selesai:]
        daftar_label_sisa = daftar_label_diulang[jumlah_selesai:]
        daftar_versi_sisa = daftar_versi[jumlah_selesai:]

        class DatasetTrainAug(Dataset):
            def __len__(self):
                return len(daftar_path_sisa)

            def __getitem__(self, idx):
                img = baca_gambar_aman(daftar_path_sisa[idx])
                if daftar_versi_sisa[idx] != 0:
                    img = augmentasi_train(img)
                pixel_values = image_processor(images=img, return_tensors="pt")["pixel_values"][0]
                return pixel_values, daftar_label_sisa[idx]

        dataset = DatasetTrainAug()
        loader = buat_dataloader_ekstraksi(dataset, shuffle=False)

        jumlah_diproses_sejak_checkpoint = 0
        for batch_pixel_values, batch_label in tqdm(
            loader, desc="Ekstraksi train", initial=jumlah_selesai // BATCH_SIZE_EKSTRAKSI,
            total=(total_semua // BATCH_SIZE_EKSTRAKSI) + 1,
        ):
            fitur = hitung_embedding_batch_aman(model, batch_pixel_values)
            daftar_embedding.append(fitur)
            daftar_label_hasil.extend(batch_label)

            jumlah_diproses_sejak_checkpoint += len(batch_label)
            if jumlah_diproses_sejak_checkpoint >= CHECKPOINT_SETIAP_N_GAMBAR:
                simpan_checkpoint(checkpoint_path, daftar_embedding, daftar_label_hasil, len(daftar_label_hasil))
                jumlah_diproses_sejak_checkpoint = 0

    embedding_array = np.concatenate(daftar_embedding, axis=0)
    label_array = np.array(daftar_label_hasil)

    np.savez(cache_path, embedding=embedding_array, label=label_array)
    print(f"Embedding train selesai, bentuk {embedding_array.shape}, disimpan ke {cache_path}")
    hapus_checkpoint_jika_ada(checkpoint_path)

    return embedding_array, label_array


# ============================================================
# Ekstraksi Embedding Validasi, Bersih Tanpa Augmentasi
# ============================================================

def ekstrak_embedding_val(df_manifest, model, image_processor, cache_path):
    if os.path.exists(cache_path):
        print(f"Cache embedding validasi ditemukan, memuat dari {cache_path}")
        data = np.load(cache_path, allow_pickle=True)
        return data["embedding"], data["label"]

    print("Mengekstrak embedding validasi")

    daftar_path = df_manifest[KOLOM_PATH_MANIFEST].tolist()
    daftar_label = [LABEL_MAP[l] for l in df_manifest[KOLOM_LABEL_MANIFEST].tolist()]
    total_semua = len(daftar_path)

    checkpoint_path, daftar_embedding, daftar_label_hasil, jumlah_selesai = muat_checkpoint_jika_ada(
        cache_path, punya_label=True
    )

    if jumlah_selesai >= total_semua:
        print("Seluruh data validasi sudah diproses sebelumnya dari checkpoint.")
    else:
        dataset = DatasetEkstraksi(
            daftar_path[jumlah_selesai:], daftar_label[jumlah_selesai:],
            transform=None, image_processor=image_processor,
        )
        loader = buat_dataloader_ekstraksi(dataset, shuffle=False)

        jumlah_diproses_sejak_checkpoint = 0
        for batch_pixel_values, batch_label in tqdm(loader, desc="Ekstraksi validasi"):
            fitur = hitung_embedding_batch_aman(model, batch_pixel_values)
            daftar_embedding.append(fitur)
            daftar_label_hasil.extend(batch_label)

            jumlah_diproses_sejak_checkpoint += len(batch_label)
            if jumlah_diproses_sejak_checkpoint >= CHECKPOINT_SETIAP_N_GAMBAR:
                simpan_checkpoint(checkpoint_path, daftar_embedding, daftar_label_hasil, len(daftar_label_hasil))
                jumlah_diproses_sejak_checkpoint = 0

    embedding_array = np.concatenate(daftar_embedding, axis=0)
    label_array = np.array(daftar_label_hasil)

    np.savez(cache_path, embedding=embedding_array, label=label_array)
    print(f"Embedding validasi selesai, bentuk {embedding_array.shape}, disimpan ke {cache_path}")
    hapus_checkpoint_jika_ada(checkpoint_path)

    return embedding_array, label_array


# ============================================================
# Ekstraksi Embedding Test, Versi Bersih dan Versi TTA
# ============================================================

def daftar_file_test_terurut(test_dir):
    file_list = [f for f in os.listdir(test_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    file_list = sorted(file_list, key=lambda x: int(os.path.splitext(x)[0]))
    return file_list


def ekstrak_embedding_test_clean(test_dir, model, image_processor, cache_path):
    if os.path.exists(cache_path):
        print(f"Cache embedding test bersih ditemukan, memuat dari {cache_path}")
        data = np.load(cache_path, allow_pickle=True)
        return data["embedding"], list(data["nama_file"])

    print("Mengekstrak embedding test, versi bersih")
    file_list = daftar_file_test_terurut(test_dir)
    daftar_path = [os.path.join(test_dir, f) for f in file_list]
    total_semua = len(daftar_path)

    checkpoint_path, daftar_embedding, _, jumlah_selesai = muat_checkpoint_jika_ada(
        cache_path, punya_label=False
    )

    if jumlah_selesai >= total_semua:
        print("Seluruh data test (bersih) sudah diproses sebelumnya dari checkpoint.")
    else:
        dataset = DatasetEkstraksi(
            daftar_path[jumlah_selesai:], daftar_label=None,
            transform=None, image_processor=image_processor,
        )
        loader = buat_dataloader_ekstraksi(dataset, shuffle=False)

        jumlah_diproses_sejak_checkpoint = 0
        jumlah_total_terkumpul = jumlah_selesai
        for batch_pixel_values, _ in tqdm(loader, desc="Ekstraksi test bersih"):
            fitur = hitung_embedding_batch_aman(model, batch_pixel_values)
            daftar_embedding.append(fitur)
            jumlah_total_terkumpul += len(fitur)

            jumlah_diproses_sejak_checkpoint += len(fitur)
            if jumlah_diproses_sejak_checkpoint >= CHECKPOINT_SETIAP_N_GAMBAR:
                simpan_checkpoint(checkpoint_path, daftar_embedding, None, jumlah_total_terkumpul)
                jumlah_diproses_sejak_checkpoint = 0

    embedding_array = np.concatenate(daftar_embedding, axis=0)
    np.savez(cache_path, embedding=embedding_array, nama_file=np.array(file_list))
    print(f"Embedding test bersih selesai, bentuk {embedding_array.shape}")
    hapus_checkpoint_jika_ada(checkpoint_path)

    return embedding_array, file_list


def ekstrak_embedding_test_tta(test_dir, model, image_processor, cache_path, n_tta):
    if os.path.exists(cache_path):
        print(f"Cache embedding test TTA ditemukan, memuat dari {cache_path}")
        data = np.load(cache_path, allow_pickle=True)
        return data["embedding"], list(data["nama_file"])

    print(f"Mengekstrak embedding test, {n_tta} view TTA per gambar")
    file_list = daftar_file_test_terurut(test_dir)
    daftar_path = [os.path.join(test_dir, f) for f in file_list]

    # Checkpoint per-view: cache_path khusus per indeks_view, jadi kalau
    # putus di tengah salah satu view, view yang sudah selesai tidak perlu
    # diulang.
    embedding_per_view = []
    for indeks_view in range(n_tta):
        cache_path_view = cache_path.replace(".npz", f"_view{indeks_view}.npz")

        if os.path.exists(cache_path_view):
            print(f"Cache TTA view {indeks_view} ditemukan, memuat dari {cache_path_view}")
            embedding_per_view.append(np.load(cache_path_view)["embedding"])
            continue

        augmentasi = dapatkan_augmentasi_tta(indeks_view)
        total_semua = len(daftar_path)

        checkpoint_path, daftar_embedding, _, jumlah_selesai = muat_checkpoint_jika_ada(
            cache_path_view, punya_label=False
        )

        if jumlah_selesai >= total_semua:
            print(f"View TTA {indeks_view} sudah diproses sebelumnya dari checkpoint.")
        else:
            dataset = DatasetEkstraksi(
                daftar_path[jumlah_selesai:], daftar_label=None,
                transform=augmentasi, image_processor=image_processor,
            )
            loader = buat_dataloader_ekstraksi(dataset, shuffle=False)

            jumlah_diproses_sejak_checkpoint = 0
            jumlah_total_terkumpul = jumlah_selesai
            for batch_pixel_values, _ in tqdm(loader, desc=f"TTA view {indeks_view}"):
                fitur = hitung_embedding_batch_aman(model, batch_pixel_values)
                daftar_embedding.append(fitur)
                jumlah_total_terkumpul += len(fitur)

                jumlah_diproses_sejak_checkpoint += len(fitur)
                if jumlah_diproses_sejak_checkpoint >= CHECKPOINT_SETIAP_N_GAMBAR:
                    simpan_checkpoint(checkpoint_path, daftar_embedding, None, jumlah_total_terkumpul)
                    jumlah_diproses_sejak_checkpoint = 0

        embedding_view = np.concatenate(daftar_embedding, axis=0)
        np.savez(cache_path_view, embedding=embedding_view)
        hapus_checkpoint_jika_ada(checkpoint_path)
        embedding_per_view.append(embedding_view)

    embedding_array = np.stack(embedding_per_view, axis=0)
    np.savez(cache_path, embedding=embedding_array, nama_file=np.array(file_list))
    print(f"Embedding test TTA selesai, bentuk {embedding_array.shape}")

    # Bersihkan file cache per-view sementara setelah hasil gabungan tersimpan
    for indeks_view in range(n_tta):
        cache_path_view = cache_path.replace(".npz", f"_view{indeks_view}.npz")
        if os.path.exists(cache_path_view):
            os.remove(cache_path_view)

    return embedding_array, file_list


# ============================================================
# Classifier MLP Probe
# ============================================================

class MLPProbeClassifier(nn.Module):
    def __init__(self, dim_embedding, hidden_dim=HIDDEN_DIM, num_classes=3, dropout=DROPOUT):
        super(MLPProbeClassifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(dim_embedding, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)


# ============================================================
# Class Weight, EarlyStopping, Training, Validasi
# ============================================================

def hitung_class_weight(label_array, num_classes=3):
    total = len(label_array)
    counts = [int((label_array == i).sum()) for i in range(num_classes)]
    weights = [total / (num_classes * c) if c > 0 else 0.0 for c in counts]
    for i, nama in enumerate(DAFTAR_LABEL_URUT):
        print(f"Kelas {nama}, jumlah {counts[i]}, bobot {weights[i]:.4f}")
    return torch.tensor(weights, dtype=torch.float32).to(DEVICE)


class EarlyStopping:
    def __init__(self, patience=PATIENCE_EARLY_STOPPING, delta=0.0):
        self.patience = patience
        self.delta = delta
        self.best_score = None
        self.counter = 0
        self.early_stop = False

    def __call__(self, current_score, model, save_path):
        if self.best_score is None or current_score > (self.best_score + self.delta):
            self.best_score = current_score
            self.counter = 0
            torch.save(model.state_dict(), save_path)
            print(f"Skor terbaik baru, Macro F1 {current_score:.4f}, model disimpan")
        else:
            self.counter += 1
            print(f"EarlyStopping counter {self.counter} dari {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
                print("Early stopping diaktifkan")


def train_satu_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    jumlah_batch = 0
    for embedding_batch, label_batch in loader:
        embedding_batch = embedding_batch.to(DEVICE)
        label_batch = label_batch.to(DEVICE)
        optimizer.zero_grad()
        output = model(embedding_batch)
        loss = criterion(output, label_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        jumlah_batch += 1
    return total_loss / jumlah_batch


def validasi_satu_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    jumlah_batch = 0
    seluruh_prediksi = []
    seluruh_label = []
    with torch.no_grad():
        for embedding_batch, label_batch in loader:
            embedding_batch = embedding_batch.to(DEVICE)
            label_batch = label_batch.to(DEVICE)
            output = model(embedding_batch)
            loss = criterion(output, label_batch)
            total_loss += loss.item()
            jumlah_batch += 1
            prediksi = torch.argmax(output, dim=1)
            seluruh_prediksi.extend(prediksi.cpu().numpy().tolist())
            seluruh_label.extend(label_batch.cpu().numpy().tolist())
    rata_loss = total_loss / jumlah_batch
    macro_f1 = f1_score(seluruh_label, seluruh_prediksi, average="macro")
    return rata_loss, macro_f1, seluruh_label, seluruh_prediksi


# ============================================================
# Prediksi Test dengan TTA
# ============================================================

def prediksi_test_dengan_tta(model, embedding_clean, embedding_tta):
    model.eval()
    seluruh_probs = []

    with torch.no_grad():
        tensor_clean = torch.tensor(embedding_clean, dtype=torch.float32).to(DEVICE)
        output_clean = model(tensor_clean)
        probs_clean = torch.softmax(output_clean, dim=1)
        seluruh_probs.append(probs_clean)

        for indeks_view in range(embedding_tta.shape[0]):
            tensor_view = torch.tensor(embedding_tta[indeks_view], dtype=torch.float32).to(DEVICE)
            output_view = model(tensor_view)
            probs_view = torch.softmax(output_view, dim=1)
            seluruh_probs.append(probs_view)

    probs_rata = torch.stack(seluruh_probs, dim=0).mean(dim=0)
    prediksi_idx = torch.argmax(probs_rata, dim=1).cpu().numpy()
    return prediksi_idx


# ============================================================
# Alur Utama
# ============================================================

def main():
    df_train = pd.read_csv(TRAIN_MANIFEST_PATH)
    df_val = pd.read_csv(VAL_MANIFEST_PATH)
    test_dir_dipakai = TEST_DIR_DRIVE

    model_siglip, processor_siglip = muat_siglip()
    image_processor = processor_siglip.image_processor

    embedding_train, label_train = ekstrak_embedding_train(
        df_train, model_siglip, image_processor, CACHE_TRAIN, N_AUG
    )
    embedding_val, label_val = ekstrak_embedding_val(
        df_val, model_siglip, image_processor, CACHE_VAL
    )
    embedding_test_clean, file_list_test = ekstrak_embedding_test_clean(
        test_dir_dipakai, model_siglip, image_processor, CACHE_TEST_CLEAN
    )
    embedding_test_tta, _ = ekstrak_embedding_test_tta(
        test_dir_dipakai, model_siglip, image_processor, CACHE_TEST_TTA, N_TTA
    )

    print("Membebaskan VRAM SigLIP setelah seluruh ekstraksi selesai")
    del model_siglip
    del processor_siglip
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    dim_embedding = embedding_train.shape[1]
    print(f"\nDimensi embedding SigLIP, {dim_embedding}")
    print(f"Total data train setelah augmentasi, {embedding_train.shape[0]}")

    dataset_train = TensorDataset(
        torch.tensor(embedding_train, dtype=torch.float32),
        torch.tensor(label_train, dtype=torch.long),
    )
    dataset_val = TensorDataset(
        torch.tensor(embedding_val, dtype=torch.float32),
        torch.tensor(label_val, dtype=torch.long),
    )

    loader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE_TRAINING, shuffle=True)
    loader_val = DataLoader(dataset_val, batch_size=BATCH_SIZE_TRAINING, shuffle=False)

    print("\nMenghitung class weight dari data train")
    class_weight = hitung_class_weight(label_train)

    model = MLPProbeClassifier(dim_embedding).to(DEVICE)
    jumlah_parameter = sum(p.numel() for p in model.parameters())
    print(f"Classifier head, jumlah parameter {jumlah_parameter}")

    criterion = nn.CrossEntropyLoss(weight=class_weight, label_smoothing=LABEL_SMOOTHING)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    early_stopping = EarlyStopping(patience=PATIENCE_EARLY_STOPPING)

    print(f"\nMemulai training MLP probe, target epoch maksimum {EPOCHS}")
    for epoch in range(1, EPOCHS + 1):
        waktu_mulai = time.time()
        rata_loss_train = train_satu_epoch(model, loader_train, criterion, optimizer)
        rata_loss_val, macro_f1_val, _, _ = validasi_satu_epoch(model, loader_val, criterion)
        durasi = time.time() - waktu_mulai

        print(
            f"Epoch {epoch} dari {EPOCHS}, loss train {rata_loss_train:.4f}, "
            f"loss val {rata_loss_val:.4f}, macro f1 val {macro_f1_val:.4f}, "
            f"durasi {durasi:.2f} detik"
        )

        early_stopping(macro_f1_val, model, BEST_MODEL_PATH)
        if early_stopping.early_stop:
            print("Training dihentikan lebih awal")
            break

    print(f"\nMemuat kembali bobot terbaik dari {BEST_MODEL_PATH}")
    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE, weights_only=True))

    _, macro_f1_final, label_final, prediksi_final = validasi_satu_epoch(model, loader_val, criterion)
    print(f"\nMacro F1 Score akhir pada validasi, nilai {macro_f1_final:.4f}")
    print("\nClassification report akhir pada validasi")
    print(classification_report(label_final, prediksi_final, target_names=DAFTAR_LABEL_URUT))

    print("\nMemprediksi data test memakai TTA")
    prediksi_test_idx = prediksi_test_dengan_tta(model, embedding_test_clean, embedding_test_tta)
    prediksi_test_label = [IDX_TO_LABEL[i] for i in prediksi_test_idx]

    df_submission = pd.DataFrame({
        "File": file_list_test,
        "Prediksi": prediksi_test_label,
    })
    df_submission.to_csv(SUBMISSION_PATH, index=False)
    print(f"\nSubmission disimpan pada {SUBMISSION_PATH}")
    print("\nDistribusi prediksi pada data test")
    print(df_submission["Prediksi"].value_counts())


if __name__ == "__main__":
    main()

Device cuda
ROOT_DIR /content/drive/MyDrive/BDC
NUM_WORKERS_EKSTRAKSI dipakai, 8
Path pada /content/drive/MyDrive/BDC/Preprocessing_Data/train_manifest.csv sudah tidak mengandung path Windows lama, remap dilewati
Path pada /content/drive/MyDrive/BDC/Preprocessing_Data/val_manifest.csv sudah tidak mengandung path Windows lama, remap dilewati
Memuat google/siglip-so400m-patch14-384


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

Vision encoder dibekukan
Cache embedding train ditemukan, memuat dari /content/drive/MyDrive/BDC/EDA/cache_embedding_siglip/embedding_train_naug2.npz
Cache embedding validasi ditemukan, memuat dari /content/drive/MyDrive/BDC/EDA/cache_embedding_siglip/embedding_val.npz
Mengekstrak embedding test, versi bersih


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Ekstraksi test bersih:   0%|          | 0/23 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_nu

Embedding test bersih selesai, bentuk (1458, 1152)
Mengekstrak embedding test, 5 view TTA per gambar


TTA view 4: 100%|██████████| 23/23 [01:22<00:00,  3.58s/it]


Embedding test TTA selesai, bentuk (5, 1458, 1152)
Membebaskan VRAM SigLIP setelah seluruh ekstraksi selesai

Dimensi embedding SigLIP, 1152
Total data train setelah augmentasi, 54308

Menghitung class weight dari data train
Kelas Recyclable, jumlah 16984, bobot 1.0659
Kelas Electronic, jumlah 15996, bobot 1.1317
Kelas Organic, jumlah 21328, bobot 0.8488
Classifier head, jumlah parameter 295939

Memulai training MLP probe, target epoch maksimum 30
Epoch 1 dari 30, loss train 0.2916, loss val 0.2269, macro f1 val 0.9851, durasi 2.54 detik
Skor terbaik baru, Macro F1 0.9851, model disimpan
Epoch 2 dari 30, loss train 0.2131, loss val 0.2234, macro f1 val 0.9855, durasi 1.69 detik
Skor terbaik baru, Macro F1 0.9855, model disimpan
Epoch 3 dari 30, loss train 0.2080, loss val 0.2222, macro f1 val 0.9857, durasi 1.39 detik
Skor terbaik baru, Macro F1 0.9857, model disimpan
Epoch 4 dari 30, loss train 0.2049, loss val 0.2218, macro f1 val 0.9857, durasi 1.40 detik
Skor terbaik baru, Macro F1